# Pharos: Fine-tune Gemma 4 E2B for Pharmaceutical Verification
Fine-tune the Gemma 4 E2B vision-language model on pharmaceutical blister pack images using Unsloth for optimized training.

In [ ]:
# Install dependencies
!pip install -q unsloth[gemma2] -U
!pip install -q torch torchvision
!pip install -q peft transformers bitsandbytes
!pip install -q pillow opencv-python
print("✓ Dependencies installed")

In [ ]:
import torch
import json
from pathlib import Path
from unsloth import FastVisionModel
from datasets import Dataset, load_dataset
from transformers import TrainingArguments, Trainer
from PIL import Image
import numpy as np

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load model with Unsloth optimization
model, processor = FastVisionModel.from_pretrained(
    model_name="unsloth/Gemma-4E2B-it-gguf",
    load_in_4bit=True,
    max_seq_length=2048,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_vision_projector=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_rslora=False,
)

print("✓ Model loaded with LoRA adaptation")

In [ ]:
# Load and prepare dataset
# Expected format: parquet file with 'image' (base64 or PIL Image) and 'label' (JSON string) columns
# Download or mount your dataset here

def load_pharos_dataset(dataset_path: str) -> Dataset:
    """
    Load pharmaceutical blister pack dataset.
    Expected CSV with columns: image_path, medicine_name, batch_code, risk_level, flags
    """
    data = []
    with open(dataset_path, 'r') as f:
        for line in f:
            record = json.loads(line)
            image_path = record.get('image_path')
            
            # Load image
            image = Image.open(image_path).convert('RGB')
            
            # Build label
            label = json.dumps({
                'medicine_name': record.get('medicine_name', 'Unknown'),
                'batch_code': record.get('batch_code', ''),
                'expiry_date': record.get('expiry_date', ''),
                'risk_score': record.get('risk_score', 50),
                'risk_level': record.get('risk_level', 'MODERATE'),
                'flags': record.get('flags', []),
                'image_quality_ok': record.get('image_quality_ok', True),
            })
            
            data.append({'image': image, 'label': label})
    
    return Dataset.from_dict({
        'image': [d['image'] for d in data],
        'label': [d['label'] for d in data]
    })

# For demonstration, create a minimal synthetic dataset
print("Note: Replace with your actual Pharos dataset (mount or download)")
print("Expected format: JSONL file with image_path and metadata fields")

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./pharos_gemma4_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=1e-4,
    warmup_steps=500,
    logging_steps=100,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=250,
    optim="adamw_8bit",
    bf16=True,
    max_grad_norm=0.3,
    seed=42,
)

print("✓ Training configuration ready")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Epochs: {training_args.num_train_epochs}")

In [ ]:
# Define collate function for batching
def collate_fn(batch):
    images = []
    labels = []
    
    for item in batch:
        image = item['image']
        label = item['label']
        
        # Process image
        if isinstance(image, str):
            image = Image.open(image).convert('RGB')
        
        images.append(image)
        labels.append(label)
    
    # Process with vision processor
    inputs = processor(images=images, text=labels, padding=True, return_tensors='pt')
    
    return inputs

print("✓ Collate function defined")

In [ ]:
# Initialize trainer (with placeholder dataset for demonstration)
# Replace train_dataset with your actual loaded dataset

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=None,  # Load with load_pharos_dataset()
    eval_dataset=None,   # Split from training data
    data_collator=collate_fn,
)

print("✓ Trainer initialized")
print("\nTo train:")
print("1. Load your Pharos dataset: dataset = load_pharos_dataset('your_data.jsonl')")
print("2. Split: train_test_split = dataset.train_test_split(test_size=0.1)")
print("3. Set trainer: trainer.train_dataset = train_test_split['train']")
print("4. Start training: trainer.train()")

In [ ]:
# Save fine-tuned model
# After training completes

def save_finetuned_model(model, processor, output_path: str = './pharos_gemma4_final'):
    """
    Save the fine-tuned model and processor for deployment.
    """
    Path(output_path).mkdir(parents=True, exist_ok=True)
    
    # Save model
    model.save_pretrained(f'{output_path}/model')
    processor.save_pretrained(f'{output_path}/processor')
    
    # Save adapter config for deployment
    with open(f'{output_path}/adapter_config.json', 'w') as f:
        json.dump({
            'base_model': 'unsloth/Gemma-4E2B-it-gguf',
            'finetune_vision_layers': True,
            'finetune_vision_projector': True,
            'max_seq_length': 2048,
            'lora_r': 16,
            'lora_alpha': 32,
            'lora_dropout': 0.05,
        }, f, indent=2)
    
    print(f"✓ Model saved to {output_path}")
    print(f"  - Model weights: {output_path}/model")
    print(f"  - Processor: {output_path}/processor")
    print(f"  - Config: {output_path}/adapter_config.json")

print("✓ save_finetuned_model function ready")

In [ ]:
# Evaluation and benchmarking
def evaluate_model(model, processor, test_dataset):
    """
    Evaluate fine-tuned model on test set.
    Returns accuracy and F1-score for risk_level prediction.
    """
    model.eval()
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in test_dataset:
            image = batch['image']
            label = json.loads(batch['label'])
            
            # Run inference
            inputs = processor(images=[image], return_tensors='pt')
            outputs = model.generate(**inputs, max_new_tokens=256)
            
            # Parse output
            prediction = processor.decode(outputs[0], skip_special_tokens=True)
            
            # Compare risk_level
            try:
                pred_json = json.loads(prediction)
                if pred_json.get('risk_level') == label.get('risk_level'):
                    correct += 1
            except:
                pass
            
            total += 1
    
    accuracy = correct / total if total > 0 else 0
    print(f"✓ Evaluation complete: {accuracy:.2%} accuracy on risk_level prediction")
    return accuracy

print("✓ evaluate_model function ready")